In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/fred-economic-agent/data/processed/economic_dataset.csv',
                  index_col='Date', parse_dates=True)
print(df.shape)
df.head()

Mounted at /content/drive
(590, 13)


,Unemployment Rate,CPI (All Items),Industrial Production Index,Federal Funds Rate,10-Year Treasury Yield,2-Year Treasury Yield,Real GDP,NBER Recession Indicator (target),Yield Curve Spread,Inflation Rate,Unemployment Change,Industrial Production Growth,Recession
Date,,,,,,,,,,,,,
1977-06-01,7.2,60.5,48.0116,5.39,7.275455,6.127273,6654.466,0.0,1.148182,6.701940,0.2,9.207369,0
1977-07-01,6.9,60.8,48.0730,5.42,7.330000,6.274737,6774.457,0.0,1.055263,6.666667,-0.3,8.727759,0
1977-08-01,7.0,61.1,48.1205,5.90,7.396522,6.611304,6774.457,0.0,0.785217,6.631763,0.1,8.091747,0
1977-09-01,6.8,61.3,48.3189,6.14,7.343810,6.714286,6774.457,0.0,0.629524,6.423611,-0.2,8.163485,0
1977-10-01,6.8,61.6,48.3871,6.47,7.523500,7.106000,6774.592,0.0,0.417500,6.390328,0.0,8.349138,0


In [2]:
# --- Feature engineering with leakage prevention ---
# We predict CURRENT month's recession status using PRIOR month's indicator values.
# This ensures no future information leaks into the prediction.

feature_cols = [
    'Unemployment Rate', 'Unemployment Change', 'Inflation Rate',
    'Industrial Production Growth', 'Yield Curve Spread', 'Federal Funds Rate'
]

model_df = df.copy()

# Shift all features by 1 month (lag-1) so we only use "already known" data
for col in feature_cols:
    model_df[f'{col}_lag1'] = model_df[col].shift(1)

lagged_feature_cols = [f'{col}_lag1' for col in feature_cols]

# Drop the first row (NaN after shifting) and keep only lagged features + target
model_df = model_df[lagged_feature_cols + ['Recession']].dropna()

print(model_df.shape)
model_df.head()

(589, 7)


,Unemployment Rate_lag1,Unemployment Change_lag1,Inflation Rate_lag1,Industrial Production Growth_lag1,Yield Curve Spread_lag1,Federal Funds Rate_lag1,Recession
Date,,,,,,,
1977-07-01,7.2,0.2,6.701940,9.207369,1.148182,5.39,0
1977-08-01,6.9,-0.3,6.666667,8.727759,1.055263,5.42,0
1977-09-01,7.0,0.1,6.631763,8.091747,0.785217,5.90,0
1977-10-01,6.8,-0.2,6.423611,8.163485,0.629524,6.14,0
1977-11-01,6.8,0.0,6.390328,8.349138,0.417500,6.47,0


In [3]:
# --- Time-based train/test split (NO shuffling — this is time-series data) ---
# Train on the earlier ~80% of history, test on the most recent ~20%.
# This mimics real deployment: predicting a future you haven't seen yet.

split_idx = int(len(model_df) * 0.8)

X = model_df[lagged_feature_cols]
y = model_df['Recession']

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Train period:", X_train.index.min(), "to", X_train.index.max())
print("Test period:", X_test.index.min(), "to", X_test.index.max())
print("\nTrain shape:", X_train.shape, " Test shape:", X_test.shape)
print("\nTrain recession rate: {:.1%}".format(y_train.mean()))
print("Test recession rate: {:.1%}".format(y_test.mean()))

Train period: 1977-07-01 00:00:00 to 2016-09-01 00:00:00
Test period: 2016-10-01 00:00:00 to 2026-07-01 00:00:00

Train shape: (471, 6)  Test shape: (118, 6)

Train recession rate: 11.9%
Test recession rate: 1.7%


In [4]:
# --- Time-Series Cross-Validation setup ---
# Instead of one fixed split (which can starve the test set of recession examples),
# we use multiple time-ordered folds so different recessions appear in different test folds.
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

# Sanity check: print recession rate in each fold's test portion
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    test_dates = X.iloc[test_idx].index
    test_recession_rate = y.iloc[test_idx].mean()
    print(f"Fold {fold}: test period {test_dates.min().date()} to {test_dates.max().date()}, "
          f"recession rate: {test_recession_rate:.1%}")

Fold 1: test period 1985-10-01 to 1993-11-01, recession rate: 8.2%
Fold 2: test period 1993-12-01 to 2002-01-01, recession rate: 8.2%
Fold 3: test period 2002-02-01 to 2010-03-01, recession rate: 18.4%
Fold 4: test period 2010-04-01 to 2018-05-01, recession rate: 0.0%
Fold 5: test period 2018-06-01 to 2026-07-01, recession rate: 2.0%


In [6]:
# --- Train models using cross-validated out-of-fold predictions ---
# Instead of averaging per-fold metrics (which breaks when a fold has 0 recessions),
# we collect predictions from each model across all folds, then score once at the end.
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# class_weight='balanced' compensates for recessions being a rare minority class (~10%)
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=200, max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
}

# --- Manual out-of-fold prediction loop (TimeSeriesSplit-safe) ---
# cross_val_predict requires every row to appear in some test fold, but with
# TimeSeriesSplit the earliest rows are only ever used for training (never tested).
# So we loop manually and only score rows that were actually held out.

oof_predictions = {}
oof_mask = None  # tracks which rows ever got a prediction

for name, model in models.items():
    preds_full = pd.Series(index=X.index, dtype='float64')  # NaN by default

    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr = y.iloc[train_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        preds_full.iloc[test_idx] = preds

    oof_predictions[name] = preds_full
    if oof_mask is None:
        oof_mask = preds_full.notna()  # rows that got at least one prediction

    print(f"{name}: done")

print("\nRows evaluated (had at least one out-of-fold prediction):", oof_mask.sum(), "out of", len(X))

Logistic Regression: done
Decision Tree: done
Random Forest: done
Gradient Boosting: done

Rows evaluated (had at least one out-of-fold prediction): 490 out of 589


In [7]:
# --- Evaluate all models on out-of-fold predictions ---
# Per the project brief: prioritize RECALL over raw accuracy, since missing an
# actual recession is more costly than an occasional false alarm.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []

for name, preds_full in oof_predictions.items():
    mask = preds_full.notna()
    y_true = y[mask]
    y_pred = preds_full[mask]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results).set_index('Model').sort_values('Recall', ascending=False)
print(results_df.round(3))

# Confusion matrices for each model
for name, preds_full in oof_predictions.items():
    mask = preds_full.notna()
    y_true = y[mask]
    y_pred = preds_full[mask]
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{name} Confusion Matrix:")
    print(f"                Predicted No-Rec   Predicted Recession")
    print(f"Actual No-Rec        {cm[0][0]:>6}              {cm[0][1]:>6}")
    print(f"Actual Recession     {cm[1][0]:>6}              {cm[1][1]:>6}")

                     Accuracy  Precision  Recall     F1
Model                                                  
Logistic Regression     0.947      0.727   0.444  0.552
Gradient Boosting       0.888      0.148   0.111  0.127
Decision Tree           0.890      0.125   0.083  0.100
Random Forest           0.912      0.111   0.028  0.044

Logistic Regression Confusion Matrix:
                Predicted No-Rec   Predicted Recession
Actual No-Rec           448                   6
Actual Recession         20                  16

Decision Tree Confusion Matrix:
                Predicted No-Rec   Predicted Recession
Actual No-Rec           433                  21
Actual Recession         33                   3

Random Forest Confusion Matrix:
                Predicted No-Rec   Predicted Recession
Actual No-Rec           446                   8
Actual Recession         35                   1

Gradient Boosting Confusion Matrix:
                Predicted No-Rec   Predicted Recession
Actual No-Rec 

In [8]:
# --- Train final Logistic Regression model on ALL available data ---
# (Cross-validation was for honest evaluation; the deployed model uses everything we have)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

final_model.fit(X, y)

# Inspect which features drive the prediction (useful later for GenAI explanations)
coef_df = pd.DataFrame({
    'Feature': lagged_feature_cols,
    'Coefficient': final_model.named_steps['clf'].coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df)

                             Feature  Coefficient
3  Industrial Production Growth_lag1    -2.167413
5            Federal Funds Rate_lag1     2.117453
4            Yield Curve Spread_lag1     2.005242
0             Unemployment Rate_lag1    -1.802807
2                Inflation Rate_lag1     1.183068
1           Unemployment Change_lag1     1.098627


In [9]:
# --- Check pairwise correlation among our lagged features to find redundancy ---
X.corr().round(2)

,Unemployment Rate_lag1,Unemployment Change_lag1,Inflation Rate_lag1,Industrial Production Growth_lag1,Yield Curve Spread_lag1,Federal Funds Rate_lag1
Unemployment Rate_lag1,1.00,0.14,0.03,-0.22,0.45,0.11
Unemployment Change_lag1,0.14,1.00,0.01,-0.22,-0.03,0.03
Inflation Rate_lag1,0.03,0.01,1.00,0.09,-0.51,0.70
Industrial Production Growth_lag1,-0.22,-0.22,0.09,1.00,-0.08,0.13
Yield Curve Spread_lag1,0.45,-0.03,-0.51,-0.08,1.00,-0.64
Federal Funds Rate_lag1,0.11,0.03,0.70,0.13,-0.64,1.00


In [10]:
# --- Drop Federal Funds Rate_lag1 due to multicollinearity with Inflation and Yield Curve Spread ---
# This should let the remaining coefficients reflect real economic relationships,
# rather than fighting each other for credit.

reduced_features = [c for c in lagged_feature_cols if c != 'Federal Funds Rate_lag1']
X_reduced = model_df[reduced_features]

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

final_model.fit(X_reduced, y)

coef_df = pd.DataFrame({
    'Feature': reduced_features,
    'Coefficient': final_model.named_steps['clf'].coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df)

                             Feature  Coefficient
2                Inflation Rate_lag1     1.858114
3  Industrial Production Growth_lag1    -1.727447
1           Unemployment Change_lag1     1.155859
4            Yield Curve Spread_lag1     0.712093
0             Unemployment Rate_lag1    -0.551819


In [11]:
# --- Final feature set: drop Unemployment Rate_lag1 (correlated with Yield Curve Spread,
# causing an economically backwards coefficient sign). Unemployment Change_lag1 is kept
# and still captures unemployment dynamics without the distortion. ---

final_features = [
    'Unemployment Change_lag1', 'Inflation Rate_lag1',
    'Industrial Production Growth_lag1', 'Yield Curve Spread_lag1'
]
X_final = model_df[final_features]

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

final_model.fit(X_final, y)

coef_df = pd.DataFrame({
    'Feature': final_features,
    'Coefficient': final_model.named_steps['clf'].coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)

                             Feature  Coefficient
2  Industrial Production Growth_lag1    -1.524890
1                Inflation Rate_lag1     1.517663
0           Unemployment Change_lag1     1.188795
3            Yield Curve Spread_lag1     0.512847


In [15]:
# --- Re-validate recall with the final reduced feature set ---
X_cv = model_df[final_features]

preds_full = pd.Series(index=X_cv.index, dtype='float64')
for train_idx, test_idx in tscv.split(X_cv):
    X_tr, X_te = X_cv.iloc[train_idx], X_cv.iloc[test_idx]
    y_tr = y.iloc[train_idx]
    model = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
    model.fit(X_tr, y_tr)
    preds_full.iloc[test_idx] = model.predict(X_te)

mask = preds_full.notna()
print("Accuracy:", round(accuracy_score(y[mask], preds_full[mask]), 3))
print("Precision:", round(precision_score(y[mask], preds_full[mask], zero_division=0), 3))
print("Recall:", round(recall_score(y[mask], preds_full[mask], zero_division=0), 3))
print("F1:", round(f1_score(y[mask], preds_full[mask], zero_division=0), 3))

Accuracy: 0.916
Precision: 0.444
Recall: 0.556
F1: 0.494


In [16]:
# --- Final check + save the recession classification model ---
import joblib
import os

# Confirm final coefficient signs before saving
coef_df = pd.DataFrame({
    'Feature': final_features,
    'Coefficient': final_model.named_steps['clf'].coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)

# Save model to Drive so later notebooks (GenAI, Agent, Streamlit) can load it
os.makedirs('/content/drive/MyDrive/fred-economic-agent/models', exist_ok=True)
joblib.dump(final_model, '/content/drive/MyDrive/fred-economic-agent/models/recession_model.pkl')

# Also save the feature list — later notebooks need to know the exact input order
joblib.dump(final_features, '/content/drive/MyDrive/fred-economic-agent/models/recession_model_features.pkl')

print("\nModel saved to models/recession_model.pkl")
print("Features used:", final_features)

                             Feature  Coefficient
2  Industrial Production Growth_lag1    -1.524890
1                Inflation Rate_lag1     1.517663
0           Unemployment Change_lag1     1.188795
3            Yield Curve Spread_lag1     0.512847

Model saved to models/recession_model.pkl
Features used: ['Unemployment Change_lag1', 'Inflation Rate_lag1', 'Industrial Production Growth_lag1', 'Yield Curve Spread_lag1']


In [17]:
# --- Correct the lag for Yield Curve Spread specifically ---
# Yield curve inversion is a LEADING indicator over ~6-18 months, not 1 month.
# Using lag-1 made it look contemporaneous (near-zero relationship), causing
# an unstable/backwards coefficient. We use lag-12 instead, matching the
# -0.33 correlation we already proved earlier in EDA.

model_df['Yield Curve Spread_lag12'] = df['Yield Curve Spread'].shift(12)

final_features_v2 = [
    'Unemployment Change_lag1', 'Inflation Rate_lag1',
    'Industrial Production Growth_lag1', 'Yield Curve Spread_lag12'
]

model_df_v2 = model_df[final_features_v2 + ['Recession']].dropna()
X_v2 = model_df_v2[final_features_v2]
y_v2 = model_df_v2['Recession']

final_model_v2 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])
final_model_v2.fit(X_v2, y_v2)

coef_df = pd.DataFrame({
    'Feature': final_features_v2,
    'Coefficient': final_model_v2.named_steps['clf'].coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)

                             Feature  Coefficient
2  Industrial Production Growth_lag1    -1.370320
3           Yield Curve Spread_lag12    -1.249342
0           Unemployment Change_lag1     1.141740
1                Inflation Rate_lag1     0.829776


In [18]:
# --- Re-validate recall with corrected features (lag-12 yield curve) ---
preds_full = pd.Series(index=X_v2.index, dtype='float64')
for train_idx, test_idx in tscv.split(X_v2):
    X_tr, X_te = X_v2.iloc[train_idx], X_v2.iloc[test_idx]
    y_tr = y_v2.iloc[train_idx]
    model = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
    model.fit(X_tr, y_tr)
    preds_full.iloc[test_idx] = model.predict(X_te)

mask = preds_full.notna()
print("Accuracy:", round(accuracy_score(y_v2[mask], preds_full[mask]), 3))
print("Precision:", round(precision_score(y_v2[mask], preds_full[mask], zero_division=0), 3))
print("Recall:", round(recall_score(y_v2[mask], preds_full[mask], zero_division=0), 3))
print("F1:", round(f1_score(y_v2[mask], preds_full[mask], zero_division=0), 3))

# Save the corrected final model
joblib.dump(final_model_v2, '/content/drive/MyDrive/fred-economic-agent/models/recession_model.pkl')
joblib.dump(final_features_v2, '/content/drive/MyDrive/fred-economic-agent/models/recession_model_features.pkl')
print("\nFinal model saved (overwriting previous version)")

Accuracy: 0.904
Precision: 0.417
Recall: 0.694
F1: 0.521

Final model saved (overwriting previous version)


ECONOMIC SNAPSHOT

Recession Risk HIGH

Inflation Trend FALLING

Unemployment Trend RISING

Industrial Production DECLINING

Yield Curve INVERTED

Anomalies 3

In [19]:
# --- Economic Intelligence Snapshot ---
# Takes the latest data + trained model, produces a structured summary.
# This structured object is what gets handed to the GenAI layer later —
# the LLM will only ever see these clean labels/numbers, never raw guesses.

def generate_economic_snapshot(df, model, feature_list, lookback_months=6):
    """
    df: full cleaned economic dataframe (with derived indicators)
    model: trained recession classification pipeline
    feature_list: exact feature names/order the model expects
    lookback_months: window used to judge "trend" (rising/falling/stable)
    """
    latest = df.iloc[-1]
    recent = df.iloc[-lookback_months:]

    # --- Build the model's input row using the SAME lag logic as training ---
    # Unemployment Change_lag1, Inflation Rate_lag1, Industrial Production Growth_lag1 -> lag 1
    # Yield Curve Spread_lag12 -> lag 12
    input_row = {
        'Unemployment Change_lag1': df['Unemployment Change'].iloc[-2],
        'Inflation Rate_lag1': df['Inflation Rate'].iloc[-2],
        'Industrial Production Growth_lag1': df['Industrial Production Growth'].iloc[-2],
        'Yield Curve Spread_lag12': df['Yield Curve Spread'].iloc[-13],
    }
    X_latest = pd.DataFrame([input_row])[feature_list]  # enforce correct column order

    recession_prob = model.predict_proba(X_latest)[0][1]

    # --- Convert probability to a business-friendly risk label ---
    if recession_prob >= 0.66:
        risk_label = "HIGH"
    elif recession_prob >= 0.33:
        risk_label = "MODERATE"
    else:
        risk_label = "LOW"

    # --- Trend helpers: compare latest value to N months ago ---
    def trend(series, threshold=0.05):
        change = series.iloc[-1] - series.iloc[0]
        if change > threshold:
            return "RISING"
        elif change < -threshold:
            return "FALLING"
        return "STABLE"

    inflation_trend = trend(recent['Inflation Rate'])
    unemployment_trend = trend(recent['Unemployment Rate'])
    ip_trend = trend(recent['Industrial Production Growth'])
    yield_curve_status = "INVERTED" if latest['Yield Curve Spread'] < 0 else "NORMAL"

    snapshot = {
        "date": df.index[-1].strftime('%Y-%m-%d'),
        "recession_risk": risk_label,
        "recession_probability": round(float(recession_prob), 3),
        "inflation_trend": inflation_trend,
        "inflation_rate": round(float(latest['Inflation Rate']), 2),
        "unemployment_trend": unemployment_trend,
        "unemployment_rate": round(float(latest['Unemployment Rate']), 2),
        "industrial_production_trend": ip_trend,
        "yield_curve_status": yield_curve_status,
        "yield_curve_spread": round(float(latest['Yield Curve Spread']), 2),
    }
    return snapshot

# Load the saved model + features (in case this is a fresh session)
final_model_v2 = joblib.load('/content/drive/MyDrive/fred-economic-agent/models/recession_model.pkl')
final_features_v2 = joblib.load('/content/drive/MyDrive/fred-economic-agent/models/recession_model_features.pkl')

snapshot = generate_economic_snapshot(df, final_model_v2, final_features_v2)
import json
print(json.dumps(snapshot, indent=2))

{
  "date": "2026-07-01",
  "recession_risk": "LOW",
  "recession_probability": 0.183,
  "inflation_trend": "RISING",
  "inflation_rate": 3.3,
  "unemployment_trend": "FALLING",
  "unemployment_rate": 4.1,
  "industrial_production_trend": "RISING",
  "yield_curve_status": "NORMAL",
  "yield_curve_spread": 0.38
}


In [20]:
# --- Save the snapshot function to src/ for reuse across notebooks ---
import os
os.makedirs('/content/drive/MyDrive/fred-economic-agent/src', exist_ok=True)

snapshot_code = '''
import pandas as pd

def generate_economic_snapshot(df, model, feature_list, lookback_months=6):
    """
    df: full cleaned economic dataframe (with derived indicators)
    model: trained recession classification pipeline
    feature_list: exact feature names/order the model expects
    lookback_months: window used to judge "trend" (rising/falling/stable)
    """
    latest = df.iloc[-1]
    recent = df.iloc[-lookback_months:]

    input_row = {
        'Unemployment Change_lag1': df['Unemployment Change'].iloc[-2],
        'Inflation Rate_lag1': df['Inflation Rate'].iloc[-2],
        'Industrial Production Growth_lag1': df['Industrial Production Growth'].iloc[-2],
        'Yield Curve Spread_lag12': df['Yield Curve Spread'].iloc[-13],
    }
    X_latest = pd.DataFrame([input_row])[feature_list]

    recession_prob = model.predict_proba(X_latest)[0][1]

    if recession_prob >= 0.66:
        risk_label = "HIGH"
    elif recession_prob >= 0.33:
        risk_label = "MODERATE"
    else:
        risk_label = "LOW"

    def trend(series, threshold=0.05):
        change = series.iloc[-1] - series.iloc[0]
        if change > threshold:
            return "RISING"
        elif change < -threshold:
            return "FALLING"
        return "STABLE"

    inflation_trend = trend(recent['Inflation Rate'])
    unemployment_trend = trend(recent['Unemployment Rate'])
    ip_trend = trend(recent['Industrial Production Growth'])
    yield_curve_status = "INVERTED" if latest['Yield Curve Spread'] < 0 else "NORMAL"

    snapshot = {
        "date": df.index[-1].strftime('%Y-%m-%d'),
        "recession_risk": risk_label,
        "recession_probability": round(float(recession_prob), 3),
        "inflation_trend": inflation_trend,
        "inflation_rate": round(float(latest['Inflation Rate']), 2),
        "unemployment_trend": unemployment_trend,
        "unemployment_rate": round(float(latest['Unemployment Rate']), 2),
        "industrial_production_trend": ip_trend,
        "yield_curve_status": yield_curve_status,
        "yield_curve_spread": round(float(latest['Yield Curve Spread']), 2),
    }
    return snapshot
'''

with open('/content/drive/MyDrive/fred-economic-agent/src/snapshot.py', 'w') as f:
    f.write(snapshot_code)

print("Saved to src/snapshot.py")

Saved to src/snapshot.py
